# Week 05 (JSON Review)

Some exercises with `JSON` files, `Python` lists, `for` loops and `if` statements.

## Setup

Run the following 2 cells to import all necessary data, libraries and helpers.

In [ ]:
!wget -q "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/main/src/data_utils.py" -O data_utils.py

In [ ]:
# import helper
from data_utils import object_from_json_url

## Data Exploration

Let's open up the MoMA art dataset to answer some questions about their art collection.

This next cell opens up the `moma_art.json` file into the `artworks` variable.

This will be a list of lists. The first sub-list has all of the dataset column names and the other rows contain information about individual artworks.

In [ ]:
MOMA_URL = "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/refs/heads/main/datasets/json/moma_art.json"
artworks = object_from_json_url(MOMA_URL)


In [ ]:
# Look at first item
artworks[0]

In [ ]:
# Look at properties
artworks[0].keys()

### Answer the following questions:

- How many artworks are in the dataset ?
- How many features/parameters/descriptors do each of the artworks have ? Or, how many columns are in the dataset?

In [ ]:
# TODO: How many artworks ?
print(len(artworks))

# TODO: How many columns ?
print(len(artworks[0]))

### Some More Questions:

- What's the tallest work in the collection ?
- What's the heaviest ?
- Widest ?

These can be answered by iterating through the list of artworks and keeping the artwork with the largest value seen in a separate variable.

In [ ]:
# NOTE: for each of these, assign the first artwork as the tallest/heaviest/widest
#       and then iterate over all works and find taller, heavier, wider ones

# TODO: Tallest ?
# TODO: Heaviest ?
# TODO: Widest ?

tallest = artworks[0]
heaviest = artworks[0]
widest = artworks[0]

for aw in artworks:
  if aw["Height (cm)"] > tallest["Height (cm)"]:
    tallest = aw

  if aw["Weight (kg)"] > heaviest["Weight (kg)"]:
      heaviest = aw

  if aw["Width (cm)"] > widest["Width (cm)"]:
      widest = aw

In [ ]:
heaviest

### Heavy Art

Now that we know who did the heaviest artwork in the collection, answer the following:

- How many artworks by this artist are in the collection ?
- How much do all of these works weigh together ?
- How much area do they take up ?
- How much space do they take up ?

These can be answered by combining an `if` statement with a loop and only counting or summing when certain conditions are met.

For the last questions, it might help to create functions that turn `Width` and `Depth` into area and `Width`, `Depth` and `Height` into volume.

In [ ]:
# NOTE: in order to answer the following, it might help to pre-filter the dataset
#       and keep only the artworks by the artist in a list.

heavy_art = [aw for aw in artworks if aw["Artist"] == heaviest["Artist"]]

# TODO: How many ? this is the length of the list.
print(len(heavy_art))

# TODO: Total weight ? this is the sum of the weight column.
heavy_weights = [aw["Weight (kg)"] for aw in heavy_art]
total_heavy_weight = sum(heavy_weights)
print(total_heavy_weight)

# TODO: Total area ? this is the sum of all areas (width * depth).

def area(AW):
  return AW["Width (cm)"] * AW["Depth (cm)"]

def volume(AW):
  return area(AW) * AW["Height (cm)"]

heavy_areas = [area(aw) for aw in heavy_art]
total_heavy_area = sum(heavy_areas)
print(total_heavy_area)

# TODO: Total volume ? this is the sum of all (width * depth * height).
heavy_volumes = [volume(aw) for aw in heavy_art]
total_heavy_volume = sum(heavy_volumes)
print(total_heavy_volume)

### How much does it cost MoMA to store these works of art ?

Wait... what ??

Well, we can't really tell from this dataset, but we can make some assumptions and use a different dataset to try to approximate what this value might be.

Here's what we'll do:
- Assume the works are stored in the MoMA building (probably false, but we can adjust later)
- Find latitude and longitude values for MoMA (or for where we think the artworks are)
- Open the `nyc_housing.json` file with `Python`
- Go through this dataset and find the listing closest to the latitude and longitude values we chose above
- Use the `PRICE` and `PROPERTYSQFT` values of the closest listing and the total area of the artworks calculated above to estimate the cost of storing these artworks.

We can convert $\text{cm}^2$ to $\text{ft}^2$ by diving the area in $\text{cm}^2$ by $929$.

In [ ]:
HOUSING_URL = "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/refs/heads/main/datasets/json/nyc_housing.json"

In [ ]:
# TODO: create a function that takes two lat/lon pairs and returns the distance between them
def dist(posA, posB):
  lat_diff = (posA[0] - posB[0]) ** 2
  lon_diff = (posA[1] - posB[1]) ** 2
  return (lat_diff + lon_diff) ** 0.5

# TODO: define a lat / lon for the presumed storage location
LAT, LON = 40.761288,-73.9790487

# TODO: open ny-housing.json
house_data = object_from_json_url(HOUSING_URL)
house_data[0]

# TODO: go through house listings and find closest listing to artworks assumed location.
#       this can be done by first assuming that the first item on the dataset is the closest
#       and then looping through all properties and finding the closest to the lat/lon define
#       above

closest = house_data[0]
closest_dist = dist((LAT,LON), (closest["LATITUDE"], closest["LONGITUDE"]))

for h in house_data:
  h_dist = dist((LAT,LON), (h["LATITUDE"], h["LONGITUDE"]))
  if h_dist < closest_dist:
    closest = h
    closest_dist = h_dist

# TODO: calculate price of storage based on price per sqft
area_ft2 = total_heavy_area / 929

total_cost = area_ft2 * closest["PRICE"] / closest["PROPERTYSQFT"]
total_cost